# MIL-CREDA on the toy set — a worked example

This notebook is a **tutorial**: it trains MIL-CREDA, the method of the paper, on the three 3-D toy families
(`lunas`, `toroide` and `rollo`) and shows each piece of a run: the bags, the two training phases, the label-free
choice of the phase-2 configuration, the latent space and the attention inside a bag. It uses the paper's own
model (`MIL_CREDA_EEG.mil_creda_model.MILCREDAModel`) and training functions; only the instance encoder changes,
since the instances here are 3-D points instead of EEG windows.

The run follows the method:

1. **The data.** Each domain is cut into *bags* of points; a bag is one band of one class and plays the role of a
   subject. Source bags carry their label; target bags are used **without label** to train (their label is read
   only to evaluate).
2. **The model.** A source encoder $\theta_s$ and a target encoder $\theta_t$ (here a small MLP each), one attention
   $\varphi$ shared by both domains, the bag kernel and a linear softmax classifier $\psi$.
3. **Phase 1, supervised in the source.** $\theta_s$, $\varphi$ and $\psi$ are trained with $\mathcal L_{\mathrm{src}}$
   (no consensus, $\gamma=0$); the kernel width $\sigma$ is then fixed at the median distance between points of
   different source bags.
4. **Phase 2, adaptation.** $\theta_t$ starts as a copy of $\theta_s$ and is trained, with the attention, on
   $\mathcal L=\mathcal L_{\mathrm{src}}+\eta_{\mathrm{glob}}\mathcal L_{\mathrm{glob}}+\eta_{\mathrm{loc}}\mathcal L_{\mathrm{loc}}$;
   $\theta_s$ and $\psi$ stay frozen.
5. **The label-free pick.** Several phase-2 configurations are trained and the one with the highest **cef** (the
   effective number of predicted classes on the unlabeled target bags) is kept, as in the paper.

**Budget.** About 6 minutes on a laptop CPU (measured on an Apple M2, 4 threads). Reduced with respect to the
paper's protocol: phase 1 is **not searched** (fixed settings inside the paper's ranges, where the paper runs 20 TPE
trials) and the phase-2 pool holds the **first 6** configurations of the paper's seeded draw (the paper trains
100); each configuration trains the paper's 100 epochs. With so few configurations and one seed, the numbers here
illustrate the procedure; they are not a result. `REPRO_SMOKE=1` runs a 1-minute wiring check.

## Setup
Works locally (repository checkout) and on Kaggle (the repository is cloned into `/kaggle/working`).

In [ ]:
import os, subprocess, sys, time
from pathlib import Path

REPO_URL = "https://github.com/Daprosero/Domain_Adaptation"
here = Path.cwd().resolve()
root = next((p for p in [here, *here.parents] if (p / "src" / "MIL_CREDA_EEG").is_dir()), None)
if root is None and Path("/kaggle").exists():
    root = Path("/kaggle/working/Domain_Adaptation")
    if not root.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(root)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mne==1.10.1", "neurodsp==2.2.1"], check=True)
if root is None:
    raise RuntimeError("run this notebook from inside the Domain_Adaptation repository or on Kaggle")
sys.path.insert(0, str(root / "src"))

import matplotlib.pyplot as plt
import numpy as np
import torch

from CREDA import toy_experiment
from MIL_CREDA_EEG.reproduce import tutorial as T

torch.set_num_threads(min(4, os.cpu_count() or 1))
STARTED = time.time()
print("repository:", root)

## The budget of this run

The only cell to touch to change how much is trained. `TOY_PHASE1` holds the phase-1 settings (learning rate,
$\ell_2$ penalty, embedding width, attention temperature, epochs); `TRIALS` and `EPOCHS2` size the phase-2 pool.

In [ ]:
SMOKE = os.environ.get("REPRO_SMOKE") == "1"
FAMILIES = T.FAMILIES
SEED = 0                                                  # the toy draw
PHASE1 = dict(T.TOY_PHASE1, epochs=5) if SMOKE else dict(T.TOY_PHASE1)
TRIALS = 1 if SMOKE else 6                                # paper: 100 configurations per unit
EPOCHS2 = 5 if SMOKE else T.PAPER_EPOCHS2                 # paper: 100 epochs

print("families:        ", list(FAMILIES))
print("phase-1 settings:", PHASE1)
print("phase-2 pool:    ", TRIALS, "configurations of", EPOCHS2, "epochs; target encoder start:", T.TOY_ARM)
print("phase-2 space:   ", {k: (f"{d.low:g}", f"{d.high:g}", "log" if d.log else "lin")
                            for k, d in T.C.space("MIL-CREDA").items()})

## The data: labeled bags in the source, unlabeled bags in the target

`MIL_CREDA_Benchmark.toy_run.build_family` draws the two domains of a family (the target is a translated, rotated
and jittered copy) and cuts each class into 27 bands; each band is one bag of about 35 points. The split is the
paper's subject-wise rule for its synthetic units (`xpilot.cross_dataset_sides`): one fifth of the source bags of
each class for test, and the target in five folds per class, one for test and four for unlabeled adaptation.

Each panel overlays the source (`x`, colored by class) and the target (`o`, grey), since that is how training sees
it: **target labels enter no loss**.

In [ ]:
UNITS = {name: T.toy_unit(name, SEED) for name in FAMILIES}
fig = plt.figure(figsize=(15, 5))
for col, name in enumerate(FAMILIES, 1):
    fam, sides = UNITS[name]["family"], UNITS[name]["sides"]
    ax = fig.add_subplot(1, len(FAMILIES), col, projection="3d")
    ax.scatter(*fam.Xs.T, c=[toy_experiment.COL_DURA[int(c)] for c in fam.ys], marker="x", s=8, alpha=0.5,
               label="source (labeled)")
    ax.scatter(*fam.Xt.T, c="0.45", marker="o", s=5, alpha=0.3, label="target (unlabeled)")
    ax.view_init(*toy_experiment.VISTA_DURA[name])
    ax.set_title(name)
    ax.legend(loc="upper left", fontsize=8)
    print(f"{name:8s} bags: source train/test {len(sides['source'])}/{len(sides['source_test'])}, "
          f"target adapt/test {len(sides['target'])}/{len(sides['target_test'])}, "
          f"points per bag {sides['source'][0].windows.shape[0]}")
fig.suptitle("The toy set in input space")
fig.tight_layout()
plt.show()

## Phase 1: supervised in the source

`tutorial.toy_phase_one` is `phase2_trace.phase_one` (the paper's phase 1) with three classes and the point
encoder: the source standardization is fitted on the source training points, then `two_phase.train_source_phase`
minimizes $\mathcal L_{\mathrm{src}}$ (the cross-entropy normalized by its supremum, plus the explicit $\ell_2$
penalty) for the given epochs with batches of eight bags, re-imposing $\lVert\mathbf v_R\rVert_1\le1$ after every
step; finally $\sigma$ is fixed at the median distance between points of different source bags (`v63_setup.between_bag_sigma`).

The phase-1 copy whose target encoder is initialized from the source one (with the target's own label-free
standardization, which absorbs the translation) predicts the target **without adaptation**.

In [ ]:
BASES = {}
print(f"{'family':9s} {'sigma':>7s} {'source test BA':>15s} {'target test BA, no adaptation':>30s}")
for name in FAMILIES:
    u = UNITS[name]
    BASES[name] = b = T.toy_phase_one(u, PHASE1)
    b["sourceTest"] = T.toy_scores(b["base"], u["sides"]["source_test"], "source")
    b["targetBefore"] = T.toy_scores(b["prepared"], u["sides"]["target_test"], "target")
    print(f"{name:9s} {b['sigma']:7.3f} {b['sourceTest']['BA']:15.3f} {b['targetBefore']['BA']:30.3f}")

model = BASES["lunas"]["base"]
print("\nthe model of `lunas`:\n", model)
for part in ("source_trunk", "target_trunk", "head"):
    print(f"parameters of {part:13s}", sum(p.numel() for p in getattr(model, part).parameters()))
print("parameters of the attention ", sum(p.numel() for p in model.attention_parameters()))

fig, ax = plt.subplots(figsize=(6, 3))
for name in FAMILIES:
    ax.plot(BASES[name]["loss"], label=name)
ax.set_xlabel("epoch")
ax.set_ylabel(r"$\mathcal{L}_{\mathrm{src}}$ (training)")
ax.set_title("Phase 1")
ax.legend()
fig.tight_layout()
plt.show()

## Phase 2: adaptation, and the label-free pick

`tutorial.toy_pool` trains the first configurations of the paper's seeded phase-2 draw (`criteria_pool.draw`:
learning rate, weight decay, $\eta_{\mathrm{glob}}$, $\eta_{\mathrm{loc}}$, $\tau_{\mathrm{loc}}$, $\gamma$ and
$\tau_{\mathrm{att}}$), each with the paper's training seed. One run is what `criteria_pool.train` does for the
paper: a short dry run counts the optimizer steps, then the full run anneals the learning rate as
$\eta_2(1+10p)^{-0.75}$, calling `phase2_trace.traced_path` with the paper's objective (`"eq39+hardloc"`: the global
term with confidence-weighted target blocks, and the local term with each target bag pulled toward the source
reference of its own pseudo-labeled class). The source encoder and the classifier are frozen and checked to stay
bit-identical; only $\theta_t$ and the attention move.

Each configuration is then scored by **cef** on the unlabeled adaptation bags; the first maximum is the pick. The
target-test balanced accuracy is printed for reading only (with three classes, chance is 1/3).

In [ ]:
POOLS = {}
for name in FAMILIES:
    t0 = time.time()
    POOLS[name] = rows = T.toy_pool(UNITS[name], BASES[name], TRIALS, EPOCHS2)
    pick = T.cef_pick(rows)
    print(f"{name} ({time.time() - t0:.0f} s)")
    for r in rows:
        mark = "  <- cef pick" if r["trial"] == pick else ""
        if r["nonFinite"]:
            print(f"  trial {r['trial']}: diverged{mark}")
        else:
            print(f"  trial {r['trial']}: cef {r['cef']:.3f}  target test BA {r['test']['BA']:.3f}{mark}")

## Results

Per family: the source test (phase-1 model), the target test without adaptation, the cef pick, and the best
configuration of the pool by its target-test BA (an **oracle** that reads target labels, an optimistic bound).

In [ ]:
print(f"{'family':9s} {'source':>7s} {'target, phase 1':>16s} {'MIL-CREDA (cef)':>16s} {'oracle':>7s}")
for name in FAMILIES:
    rows = [r for r in POOLS[name] if not r["nonFinite"]]
    picked = POOLS[name][T.cef_pick(POOLS[name])]
    oracle = max(r["test"]["BA"] for r in rows)
    b = BASES[name]
    print(f"{name:9s} {b['sourceTest']['BA']:7.3f} {b['targetBefore']['BA']:16.3f} "
          f"{picked['test']['BA']:16.3f} {oracle:7.3f}")

## The latent space

The bag representations $\mathbf z_u$ (the classifier's input) of both domains, projected to 2-D by principal
components fitted on the source: circles for the source ($\theta_s$), crosses for the target ($\theta_t$), colored
by their true class. Top row: the target encoder at the start of phase 2; bottom row: after the cef pick.

In [ ]:
fig, axes = plt.subplots(2, len(FAMILIES), figsize=(4.5 * len(FAMILIES), 8), squeeze=False)
for col, name in enumerate(FAMILIES):
    sides = UNITS[name]["sides"]
    src, tgt = sides["source"] + sides["source_test"], sides["target"] + sides["target_test"]
    picked = POOLS[name][T.cef_pick(POOLS[name])]["model"]
    zs = T.representations(BASES[name]["base"], src, "source")
    center = zs.mean(0)
    _, _, axes_pca = np.linalg.svd(zs - center, full_matrices=False)
    for row, (model, when) in enumerate(((BASES[name]["prepared"], "start of phase 2"), (picked, "cef pick"))):
        ax = axes[row, col]
        for bags, z, marker in ((src, zs, "o"), (tgt, T.representations(model, tgt, "target"), "x")):
            plane = (z - center) @ axes_pca[:2].T
            ax.scatter(plane[:, 0], plane[:, 1], marker=marker, s=22,
                       c=[toy_experiment.COL_DURA[int(b.label)] for b in bags])
        ax.set_title(f"{name}: {when}")
fig.tight_layout()
plt.show()

## The attention inside a bag

The weights $\beta$ of the points of one target test bag after the cef pick (they sum to one): the larger and
darker the point, the more it contributes to the bag representation. The relevance term reads each point alone;
the consensus term ($\gamma>0$) favors points close, in the learned geometry, to the rest of their bag.

In [ ]:
fig = plt.figure(figsize=(15, 4.5))
for col, name in enumerate(FAMILIES, 1):
    bag = UNITS[name]["sides"]["target_test"][0]
    picked = POOLS[name][T.cef_pick(POOLS[name])]
    w = T.attention(picked["model"], bag)
    ax = fig.add_subplot(1, len(FAMILIES), col, projection="3d")
    x = bag.windows.numpy()
    ax.scatter(*x.T, s=2000 * w, c=w, cmap="viridis")
    ax.view_init(*toy_experiment.VISTA_DURA[name])
    ax.set_title(f"{name}: {bag.subject}, gamma={picked['params']['gamma']:.2f}, "
                 f"tau_att={picked['params']['tau_att']:.2f}\nbeta from {w.min():.3f} to {w.max():.3f}")
fig.tight_layout()
plt.show()
print(f"total runtime: {(time.time() - STARTED) / 60:.1f} min")

## Notes

* **What is the paper's and what is the tutorial's.** The model, both training phases, the objective, the phase-2
  space and sampler, the training seeds, the learning-rate schedule and the cef pick are the paper's code. The point
  encoder (`tutorial.PointTrunk`), the fixed phase-1 settings, the target encoder start with its own standardization
  (`"copy-own"`) and the pool size are this tutorial's.
* **What does not change.** Phase 1 reads only source labels; in phase 2 the source encoder and the classifier stay
  frozen (the trainer refuses to finish if they moved), $\sigma$ stays at its phase-1 value, and no target label
  enters the objective or the pick.
* **Reading the numbers.** Each target test role holds 18 bags (6 per class), so a balanced accuracy moves in steps
  of 1/18. With one seed and six configurations the gain of adaptation can go either way; the paper's conclusions
  rest on 100 configurations per unit and on the EEG data (`MIL-CREDA-EEG/Notebooks`).
* This notebook writes nothing to disk.